# Mốc 8, bước 2: kiểm tra convert model đọc chữ ONNX → TFLite

Model: `latin_PP-OCRv5_rec_mobile` (PaddleOCR, bản ONNX của RapidOCR). Mục tiêu: chắc chắn đường **ONNX → TFLite (onnx2tf)** chạy đúng **trước khi** fine-tune tiếng Việt.

Kiểm tra: TFLite và ONNX cho ra **cùng chữ** trên vài ảnh dòng chữ (lệch số nhỏ thì được, chữ đọc ra phải giống hệt).

Chạy lần lượt từng ô (Runtime → Run all cũng được). Không cần GPU cho notebook này.

In [ ]:
# 1. Cài công cụ (khoảng 1-2 phút)
!pip -q install onnx2tf onnx onnxruntime onnxsim onnx_graphsurgeon sng4onnx tf_keras psutil ai-edge-litert

In [ ]:
# 2. Tải model và từ điển ký tự (nằm sẵn trong metadata của file ONNX)
import urllib.request, onnxruntime as ort
URL = "https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.10.0/onnx/PP-OCRv5/rec/latin_PP-OCRv5_rec_mobile.onnx"
urllib.request.urlretrieve(URL, "rec.onnx")
sess = ort.InferenceSession("rec.onnx", providers=["CPUExecutionProvider"])
# Từ điển: các ký tự trong metadata + ký tự khoảng trắng ở cuối (giống RapidOCR / PaddleOCR use_space_char)
chars = sess.get_modelmeta().custom_metadata_map["character"].splitlines() + [" "]
print("vao:", [(i.name, i.shape) for i in sess.get_inputs()], "| ra:", [(o.name, o.shape) for o in sess.get_outputs()])
print("so ky tu trong tu dien:", len(chars))

In [ ]:
# 3. Convert sang TFLite. Cố định kích thước đầu vào: cao 48, rộng 320 (app sẽ đệm dòng chữ cho đủ 320)
!onnx2tf -i rec.onnx -o tflite_out -ois x:1,3,48,320 -n
!ls -la tflite_out

In [ ]:
# 4. Tạo vài ảnh dòng chữ để thử (vẽ chữ bằng PIL, giống cách PaddleOCR chuẩn bị ảnh)
import numpy as np
from PIL import Image, ImageDraw, ImageFont
!wget -q -O font.ttf https://github.com/google/fonts/raw/main/ofl/roboto/Roboto%5Bwdth,wght%5D.ttf
font = ImageFont.truetype("font.ttf", 36)
texts = ["HOA DON THANH TOAN", "So HD: 140064", "Total 50,000 d", "GIA LOC CONSTRUCTION"]

def make_line(t):
    img = Image.new("RGB", (640, 48), "white")
    ImageDraw.Draw(img).text((6, 4), t, fill="black", font=font)
    w = int(ImageDraw.Draw(img).textlength(t, font=font)) + 12
    return np.array(img)[:, :min(w, 640)]

def prep(rgb):
    # Giống RapidOCR: co về cao 48 (giữ tỉ lệ), chuẩn hoá (x/255 - 0.5)/0.5, đệm 0 cho đủ rộng 320
    h, w = rgb.shape[:2]
    nw = min(320, int(np.ceil(48 * w / h)))
    im = np.array(Image.fromarray(rgb).resize((nw, 48), Image.BILINEAR)).astype(np.float32)
    im = (im / 255.0 - 0.5) / 0.5
    out = np.zeros((48, 320, 3), np.float32)
    out[:, :nw] = im
    return out.transpose(2, 0, 1)[None]          # NCHW cho ONNX

def ctc(prob):
    # Giải mã CTC: lấy ký tự điểm cao nhất mỗi lát, gộp ký tự lặp, bỏ ký tự trống (số 0)
    idx = prob.argmax(-1)
    res, prev = [], -1
    for k in idx:
        if k != prev and k != 0:
            res.append(chars[k - 1])
        prev = k
    return "".join(res)
lines = [make_line(t) for t in texts]
print([l.shape for l in lines])

In [ ]:
# 5. So ONNX với TFLite
import glob
from ai_edge_litert.interpreter import Interpreter
path = sorted(glob.glob("tflite_out/*float32.tflite"))[0]
it = Interpreter(model_path=path); it.allocate_tensors()
din, dout = it.get_input_details()[0], it.get_output_details()[0]
print("TFLite:", path, "vao", din["shape"], "ra", dout["shape"])
for t, line in zip(texts, lines):
    x = prep(line)
    o_onnx = sess.run(None, {"x": x})[0][0]
    it.set_tensor(din["index"], x.transpose(0, 2, 3, 1).copy())   # TFLite dùng NHWC
    it.invoke()
    o_tfl = it.get_tensor(dout["index"])[0]
    same = (o_onnx.argmax(-1) == o_tfl.argmax(-1)).mean() * 100
    print(f"{t!r:28} | ONNX: {ctc(o_onnx)!r:28} | TFLite: {ctc(o_tfl)!r:28} | lech max {abs(o_onnx - o_tfl).max():.1e} | lat trung {same:.0f}%")

In [ ]:
# 6. Tải file TFLite về máy (để dùng ở bước sau)
from google.colab import files
files.download(path)